# 1. Baseline: data multiplier and one-dimensional simulation

Reproduces the **Baseline configuration** and transaction-accounting appendix in [the report](../../markdowns/eip8279_floor_calibration_report.md).

The baseline uses a floor rate of 64 gas per counted byte and CPSB = 1,530. Execution and state have separate accounting branches but share one EIP-1559 fee, updated from the larger included branch. We use the report's dated metering schedule and existing model kernels.

Run 01–04 in order. Default execution rebuilds calibrations and simulations from local source panels. Set `ONE_DIMENSIONAL_REFRESH_XATU=1` to fetch missing Xatu source chunks; set `ONE_DIMENSIONAL_REUSE_OUTPUTS=1` to inspect an existing complete run. The [README](README.md) maps every report section and figure to a notebook.

## Upstream inputs

Run the resource-demand workflow 01–04, EIP-7999 equilibrium workflow 01–02, and EIP-7999 simulation workflow 01–04 first. These supply the February–May 2026 accounting anchors, full elasticity vectors, 6,000 sampled blocks and exact transaction membership, runtime BAL, and the canonical 32 workload paths. The EIP-7999 comparison requires the outputs regenerated with the updated execution schedule. We verify that its execution multiplier matches the one-dimensional calibration before comparing mechanisms.

The RPC/static-data and BAL-carrier refresh procedures are in those upstream notebooks. This notebook adds the shared-fee-specific Xatu queries below; credentials are read from the repository's `.env`.

In [ ]:
from pathlib import Path
import os
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display, Image, Markdown

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src/shared_fee/replay.py").is_file())
for directory in (ROOT / "src", ROOT / "scripts", ROOT / "scripts/shared_fee"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))
from publication_workflow import (
    DATA, REPORT, LABELS, baseline_anchor, read_table, require_files,
    run_stage, shared_display, source_snapshot, verify_sources,
)
DATA.mkdir(parents=True, exist_ok=True)
(ROOT / "plots").mkdir(exist_ok=True)
REUSE = os.environ.get("ONE_DIMENSIONAL_REUSE_OUTPUTS", "0") == "1"
REFRESH_XATU = os.environ.get("ONE_DIMENSIONAL_REFRESH_XATU", "0") == "1"
before = source_snapshot()
pd.set_option("display.max_columns", 24)
print("Repository:", ROOT)
print("Reuse generated outputs:", REUSE, "| Refresh Xatu inputs:", REFRESH_XATU)

In [ ]:
TAG = "2026-02-01_2026-06-01"
upstream = [
    ROOT / f"data/calibration_xatu_bal_runtime_8279_blocks_{TAG}.csv",
    ROOT / f"data/calibration_rpc_static_data_transactions_{TAG}.parquet",
    ROOT / f"data/calibration_xatu_bal_runtime_8279_carrier_panel_{TAG}.parquet",
    ROOT / f"data/execution_repricing_daily_{TAG}.csv",
    ROOT / "data/glamsterdam/anchor_accounting_panel_2026-02-01_2026-05-31.csv",
    ROOT / "data/glamsterdam/equilibrium_anchor.csv",
    ROOT / "data/7999/bal_decomposition_demand_parameters.csv",
]
require_files(upstream)
if REFRESH_XATU and REUSE:
    raise ValueError("Choose source refresh or output reuse, not both.")
if REFRESH_XATU:
    # CLICKHOUSE_USER / CLICKHOUSE_PASSWORD; optional RAW_HOST and PORT.
    # Chunked membership, opcode, and runtime pulls resume completed chunks.
    run_stage("pull_transaction_inputs.py", "--chunk-size", "100",
              outputs=["transaction_inputs_6000_blocks.parquet"])
    run_stage("pull_execution_repricing_components.py", "--chunk-size", "100",
              outputs=["execution_repricing_components_6000_blocks.parquet"])
    run_stage("pull_runtime_components.py", "--chunk-size", "200",
              outputs=["runtime_components_6000_blocks.parquet"])
    run_stage("rebuild_execution_repricing.py", "--refresh-refunds",
              outputs=[f"execution_repricing_daily_eip8038_current_{TAG}.csv"])
else:
    print("Using existing Xatu inputs. Enable ONE_DIMENSIONAL_REFRESH_XATU for the pulls above.")
run_stage("rebuild_execution_repricing.py",
          outputs=[f"execution_repricing_daily_eip8038_current_{TAG}.csv"], reuse=REUSE)
run_stage("build_floor_calibration.py", outputs=[
    "transaction_floor_panel_6000_blocks.parquet",
    "equilibrium_anchor_8279.csv", "eip8279_data_multiplier.json",
], reuse=REUSE)

## Transaction-floor calibration at 64 gas per byte

For each transaction, the regular branch is the maximum of repriced execution and the static-plus-runtime-BAL floor. EIP-8037 state gas remains in its separate branch. Incremental floor gas is aggregated by sampled block, expanded using each day's block count, and divided by historical data gas. The calibrated data multiplier adds this uplift to the preceding data-metering baseline.

The calculation uses 6,000 deterministic blocks (50 per day over 120 days), containing 1,899,748 transactions. Empty sampled blocks remain in the block denominator. The following cells calculate sample size, affected transactions, uplift composition, and the multiplier from the reconstructed panel.

The later floor sweep writes generic calibration summaries for its detailed 96-gas case. We therefore select the rate-specific 64-gas anchor explicitly when revisiting this notebook after Notebook 2.

In [ ]:
base = baseline_anchor()
panel = pd.read_parquet(DATA / "transaction_floor_panel_6000_blocks.parquet", columns=[
    "block_number", "tx_index", "tx_hash", "floor_class", "affected_by_8279",
    "incremental_data_gas_8279",
])
blocks = pd.read_csv(upstream[0])
assert len(blocks) == 6000 and len(panel) == 1_899_748
assert not panel.duplicated(["block_number", "tx_index", "tx_hash"]).any()
coverage = panel.groupby("floor_class", sort=False).agg(
    transactions=("tx_hash", "size"), incremental_gas=("incremental_data_gas_8279", "sum"))
coverage["uplift_share_percent"] = 100 * coverage.incremental_gas / coverage.incremental_gas.sum()
display(pd.DataFrame([{
    "sampled_blocks": len(blocks), "sampled_transactions": len(panel),
    "affected_transactions_percent": 100 * panel.affected_by_8279.mean(),
    "data_multiplier_at_64": base.m_data, "execution_multiplier": base.m_execution,
}]))
display(coverage)
assert np.isclose(base.m_data, 2.1615, atol=0.00005)
assert np.isclose(100 * panel.affected_by_8279.mean(), 2.04648, atol=0.00001)
demand = pd.read_csv(ROOT / "data/7999/bal_decomposition_demand_parameters.csv").iloc[0]
np.testing.assert_allclose(base.m_execution, demand.m_execution, rtol=0, atol=1e-12)
del panel

## Physical limits and central replay

At each propagation allocation, the common limit is the smaller of execution capacity and the payload budget times 64 gas per byte. The target is half that limit. The historical state multiplier is unchanged.

$$
L_{\mathrm{shared}}(t)=\min\!\left[100\mathrm{M}(9-t),\,64B_{\max}(t)\right],
\qquad T_{\mathrm{shared}}=L_{\mathrm{shared}}/2.
$$

The replay uses the full 35-day elasticity vector, 32 identical multiscale workload paths, a 7,200-block burn-in, and 50,400 measured blocks per path. It solves each unshocked equilibrium before simulation. The notebook calls the existing equilibrium and replay kernels directly. In reuse mode, it selects the corresponding baseline rows from the complete factorial sweep.

In [ ]:
from run_multiscale_design_surface import (
    BURN_IN, MEASURE_BLOCKS, N_SEEDS, EPS, build_canonical_workload,
)
from shared_fee.equilibrium import SharedFeeAnchor, solve_shared_fee_equilibrium
from shared_fee.replay import SharedFeeConfig, run_shared_fee_batch
from shared_fee.optimization import physical_capacities, BLOCKS_PER_YEAR, BYTES_PER_GIB

times = [3., 3.5, 4., 4.5, 5.]
limits = np.array([min(physical_capacities(t)["execution_capacity_gas"],
                       64 * physical_capacities(t)["safe_payload_bytes"]) for t in times])
anchor = SharedFeeAnchor(
    base.q_execution_per_block, base.q_data_per_block, base.q_state_per_block,
    base.m_execution, base.m_data, base.m_state, base.base_fee_ref_gwei,
    EPS["execution"], EPS["data"], EPS["state"])
equilibria = [solve_shared_fee_equilibrium(limit / 2, anchor) for limit in limits]

if REUSE:
    surface = read_table("shared_fee_factorial_scenarios.csv")
    baseline = surface[surface.benchmark.eq("proposal_faithful") & surface.floor_payload_feasible]
    baseline = baseline.loc[baseline.groupby("propagation_time_s").shared_limit.idxmax()]
    baseline = baseline.sort_values("propagation_time_s").reset_index(drop=True)
else:
    workload = build_canonical_workload().paths
    assert workload.shape == (32, 57_600, 4)
    repeat = lambda values: np.repeat(np.asarray(values, dtype=float), N_SEEDS)
    config = SharedFeeConfig(
        gas_target=repeat(limits / 2), gas_limit=repeat(limits),
        eps_execution=EPS["execution"], eps_data=EPS["data"], eps_state=EPS["state"],
        m_execution=base.m_execution, m_data=base.m_data, m_state=base.m_state,
        q_execution_0=base.q_execution_per_block, q_data_0=base.q_data_per_block,
        q_state_0=base.q_state_per_block, p0_gwei=base.base_fee_ref_gwei,
        w_execution=demand.w_execution_reference, w_state=demand.w_state_reference,
    )
    result = run_shared_fee_batch(
        config, workload, repeat([eq.base_fee_wei for eq in equilibria]), burn_in=BURN_IN)
    records = []
    for i, (t, limit, eq) in enumerate(zip(times, limits, equilibria)):
        ix = slice(i * N_SEEDS, (i + 1) * N_SEEDS)
        record = dict(propagation_time_s=t, shared_limit=limit, shared_target=limit / 2,
                      floor_rate=64, cpsb=1530, m_data=base.m_data, m_execution=base.m_execution,
                      equilibrium_fee_wei=eq.base_fee_wei)
        for column, key in {
            "included_execution_metered": "mean_included_execution",
            "included_data_metered": "mean_included_data",
            "included_state_metered": "mean_included_state",
            "regular_binding_fraction": "regular_binding_fraction",
        }.items():
            record[column] = result[key][ix].mean()
        for column, key in {
            "shared_limit_hit_fraction": "included_limit_fraction",
            "shared_fee_log_return_sd": "log_return_sd",
        }.items():
            record[column] = result[key][ix, 1].mean()
        record["annualized_state_growth_gib"] = (
            record["included_state_metered"] / 1530 * BLOCKS_PER_YEAR / BYTES_PER_GIB)
        records.append(record)
    baseline = pd.DataFrame(records)
    baseline.to_csv(DATA / "publication_baseline_outcomes.csv", index=False)
    del result, workload
np.testing.assert_allclose(baseline.shared_limit, limits, rtol=1e-12)
np.testing.assert_allclose(baseline.equilibrium_fee_wei,
                           [eq.base_fee_wei for eq in equilibria], rtol=1e-12)
np.testing.assert_allclose(baseline.m_data, base.m_data, rtol=1e-12)
display(shared_display(baseline).round(4))
np.testing.assert_allclose(baseline.included_execution_metered / 1e6,
                           [82.0, 87.9, 92.6, 89.1, 85.3], atol=0.05, rtol=0)

## Handoff

Notebook 2 recalculates each adjusted floor and its data multiplier, runs the complete common-limit/floor/CPSB grid, and reproduces the baseline and EIP-8368 state-tail and elasticity results together. Notebook 3 adds the EIP-8372 calibration. Notebook 4 assembles all cross-mechanism tables and figures.

The standalone baseline replay writes `data/shared_fee/publication_baseline_outcomes.csv`; reuse mode reads the same configurations from the existing factorial surface.

In [ ]:
verify_sources(before)